# Verbindung zwischen MySQL und Python
Ausführliche Doku unter https://pynative.com/python-mysql-tutorial/


### MySQL Server starten
Starten eine MySQLDocker Instanz:
- docker run --name mysqltest -p 23306:3306 -e MYSQL_ROOT_PASSWORD=BDinf -e MYSQL_ROOT_HOST=% -d mysql:latest


In [4]:
!docker run --name mysqltest -p 23306:3306 -e MYSQL_ROOT_PASSWORD=BDinf -e MYSQL_ROOT_HOST=% -d mysql:latest

55b42012b090d6a30fd0f8a609eca9922de325af466b71ac39a3f9f951a85d01


In [5]:
!docker ps

CONTAINER ID   IMAGE          COMMAND                  CREATED         STATUS                  PORTS                                NAMES
55b42012b090   mysql:latest   "docker-entrypoint.sâ€¦"   3 seconds ago   Up Less than a second   33060/tcp, 0.0.0.0:23306->3306/tcp   mysqltest


### Pakete importieren

In [8]:
try:
    import mysql.connector
except:
    !pip install mysql-connector-python
    import mysql.connector
from mysql.connector import Error


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/15.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/15.4 MB 262.6 kB/s eta 0:00:59
   ---------------------------------------- 0.1/15.4 MB 491.5 kB/s eta 0:00:32
    --------------------------------------- 0.2/15.4 MB 1.1 MB/s eta 0:00:14
   - -------------------------------------- 0.5/15.4 MB 2.1 MB/s eta 0:00:08
   -- ------------------------------------- 0.8/15.4 MB 2.8 MB/s eta 0:00:06
   -- ------------------------------------- 1.0/15.4 MB 3.2 MB/s eta 0:00:05
   --- ------------------------------------ 1.3/15.4 MB 3.4 MB/s eta 0:00:05
   --- ------------------------------------ 1.5/15.4 MB 3.6 MB/s eta 0:00:04
   ---- ----------------------------------- 1.6/15.4 MB 3.4 MB/s eta 0:00:05
   ---- -------

### Datenbank electronic anlegen
- Schritt 1: Verbindung aufbauen
- Schritt 2: Datenbank anlegen
- Schritt 3: Verbindung schließen

In [9]:
# prepare database
# create database 'electronic" if not existing yet (otherwise an error is shown)

connection = mysql.connector.connect(host="localhost", port="23306", user="root",passwd="BDinf")

c = connection.cursor()
sql = 'CREATE DATABASE electronic'
c.execute(sql)

c.close()
connection.close()


### Verbindung mit Datenbank electronic herstellen
- Schritt 1: Verbindung aufbauen
- Achtung: es muss eine leere Datenbank mit dem Namen 'electronic' bereits existieren

In [10]:
connection = mysql.connector.connect(host='localhost',
                                     port='23306',
                                     database='electronic',
                                     user='root',
                                     password='BDinf')

if connection.is_connected():
    
    # Verbindung ist hergestellt
    db_Info = connection.get_server_info()
    print("Connected to MySQL Server version ", db_Info)
    

Connected to MySQL Server version  8.3.0


- Schritt 2: tue etwas (Name der verbundenen Datenbank abfragen)

In [11]:
if connection.is_connected():
    
    # Cursor öffnen für Abfragen
    cursor = connection.cursor()
    cursor.execute("select database();")
    record = cursor.fetchone()
    print("Your connected to database: ", record)


Your connected to database:  ('electronic',)


- Schritt 3: am Ende wieder schließen

In [ ]:
if connection.is_connected():
      
    # geöffnete Verbindungen wieder schließen
    cursor.close()
    connection.close()
    print("MySQL connection is closed")
    

### Create table

In [12]:
# setzt geöffnet DB-Verbindung voraus:
if connection.is_connected():
    mySql_Create_Table_Query = """CREATE TABLE Laptop ( 
                             Id int(11) NOT NULL,
                             Name varchar(250) NOT NULL,
                             Price float NOT NULL,
                             Purchase_date Date NOT NULL,
                             PRIMARY KEY (Id)) """

    cursor = connection.cursor()
    result = cursor.execute(mySql_Create_Table_Query)
    print("Laptop Table created successfully ")
    
else:
    print("Your are not connected!")
    

Laptop Table created successfully 


### insert table
- einfache Variante

In [13]:
# setzt geöffnet DB-Verbindung voraus:
if connection.is_connected():
    mySql_insert_query = """INSERT INTO Laptop (Id, Name, Price, Purchase_date) 
                           VALUES 
                           (1, 'Lenovo ThinkPad P71', 6459, '2019-08-14') """

    cursor = connection.cursor()
    result = cursor.execute(mySql_insert_query)
    connection.commit()
    print("Record inserted successfully into Laptop table")
    

Record inserted successfully into Laptop table


- mit Prepared Statements

In [14]:
# setzt geöffnet DB-Verbindung voraus:

def insertVariblesIntoTable(id, name, price, purchase_date):
    if connection.is_connected():
        cursor = connection.cursor()
        mySql_insert_query = """INSERT INTO Laptop (Id, Name, Price, Purchase_date) 
                                    VALUES (%s, %s, %s, %s) """

        recordTuple = (id, name, price, purchase_date)
        cursor.execute(mySql_insert_query, recordTuple)
        connection.commit()
        print("Record inserted successfully into Laptop table")
        
insertVariblesIntoTable(2, 'Area 51M', 6999, '2019-04-14')
insertVariblesIntoTable(3, 'MacBook Pro', 2499, '2019-06-20')


Record inserted successfully into Laptop table
Record inserted successfully into Laptop table


- Multiple Row Insert

In [15]:
# setzt geöffnet DB-Verbindung voraus:

mySql_insert_query = """INSERT INTO Laptop (Id, Name, Price, Purchase_date) 
                       VALUES (%s, %s, %s, %s) """

records_to_insert = [(4, 'HP Pavilion Power', 1999, '2019-01-11'),
                     (5, 'MSI WS75 9TL-496', 5799, '2019-02-27'),
                     (6, 'Microsoft Surface', 2330, '2019-07-23')]

cursor = connection.cursor()
cursor.executemany(mySql_insert_query, records_to_insert)
connection.commit()
print(cursor.rowcount, "Record inserted successfully into Laptop table")


3 Record inserted successfully into Laptop table


### Fehlermanagement
```python
try:
    # versuche etwas
except mysql.connector.Error as error:
    # Fehlerbehandlung
finally:
    # finally ist optional und wird immer ausgeführt (nach Fehler oder nicht Fehler)
    # hier z.B. DB wieder schließen
```
- Beispiel mit INSERT:

In [16]:
# setzt geöffnet DB-Verbindung voraus:

try:
    mySql_insert_query = """INSERT INTO Laptop (Id, Name, Price, Purchase_date) 
                           VALUES (%s, %s, %s, %s) """

    records_to_insert = [(6, 'Apple MacBook', 1999, '2019-07-11')]

    cursor = connection.cursor()
    cursor.executemany(mySql_insert_query, records_to_insert)
    connection.commit()
    print(cursor.rowcount, "Record inserted successfully into Laptop table")

except mysql.connector.Error as error:
    # bei Bedarf ein Rollback machen
    connection.rollback()
    print("Failed to insert record into MySQL table {}".format(error))


Failed to insert record into MySQL table 1062 (23000): Duplicate entry '6' for key 'Laptop.PRIMARY'


### select
- Define the SELECT statement query. Here you need to know the table, and it’s column details.
- Execute the SELECT query using the cursor.execute() method.
- Get resultSet from the cursor object using a cursor.fetchall().
- Iterate over the ResultSet and get each row and its column value.
- Close the Python database connection.
- Catch any SQL exceptions that may come up during the process.

In [17]:
# setzt geöffnet DB-Verbindung voraus:

try:
    sql_select_Query = "select * from Laptop"
    cursor = connection.cursor()
    cursor.execute(sql_select_Query)
    records = cursor.fetchall()
    print("Total number of rows in Laptop is: ", cursor.rowcount)

    print("\nPrinting each laptop record")
    for row in records:
        print("Id = ", row[0], )
        print("Name = ", row[1])
        print("Price  = ", row[2])
        print("Purchase date  = ", row[3], "\n")

except Error as e:
    print("Error reading data from MySQL table", e)


Total number of rows in Laptop is:  6

Printing each laptop record
Id =  1
Name =  Lenovo ThinkPad P71
Price  =  6459.0
Purchase date  =  2019-08-14 

Id =  2
Name =  Area 51M
Price  =  6999.0
Purchase date  =  2019-04-14 

Id =  3
Name =  MacBook Pro
Price  =  2499.0
Purchase date  =  2019-06-20 

Id =  4
Name =  HP Pavilion Power
Price  =  1999.0
Purchase date  =  2019-01-11 

Id =  5
Name =  MSI WS75 9TL-496
Price  =  5799.0
Purchase date  =  2019-02-27 

Id =  6
Name =  Microsoft Surface
Price  =  2330.0
Purchase date  =  2019-07-23 



Note:  To fetch data use cursor.execute() to run a query.
- cursor.fetchall() to fetch all rows .
- cursor.fetchone() to fetch single row.
- cursor.fetchmany(SIZE) to fetch limited rows.

SELECT mit Variablen als Parameter

In [ ]:
# setzt geöffnet DB-Verbindung voraus:

def getLaptopDetail(id):
    try:
        cursor = connection.cursor(buffered=True)
        sql_select_query = """select * from Laptop where id = %s"""
        cursor.execute(sql_select_query, (id,))
        record = cursor.fetchall()

        for row in record:
            print("Id = ", row[0], )
            print("Name = ", row[1])
            print("Join Date = ", row[2])
            print("Salary  = ", row[3], "\n")

    except mysql.connector.Error as error:
        print("Failed to get record from MySQL table: {}".format(error))


id1 = 1
id2 = 2
getLaptopDetail(id1)
getLaptopDetail(id2)


### select mit Pandas

In [ ]:
import pandas as pd

if connection.is_connected():
    print('connected')
    query = 'select * from Laptop'
    df_mysql = pd.read_sql(query, con=connection)
df_mysql


### update table

In [ ]:
# setzt geöffnet DB-Verbindung voraus:

try:
    cursor = connection.cursor()

    # multiple records to be updated in tuple format
    sql_update_query = """Update Laptop set Price = %s where id = %s"""
    records_to_update = [(3000, 3), (2750, 4)]
    cursor.executemany(sql_update_query, records_to_update)
    print(cursor.rowcount, "Records of laptop table updated successfully")

except mysql.connector.Error as error:
    print("Failed to update table record: {}".format(error))
    

### delete table

In [18]:
# setzt geöffnet DB-Verbindung voraus:

try:
    cursor = connection.cursor()

    # delete single record now
    sql_Delete_query = """Delete from Laptop where id = 6"""
    cursor.execute(sql_Delete_query)
    connection.commit()
    print("Record deleted successfully ")

except mysql.connector.Error as error:
    print("Failed to update table record: {}".format(error))
    

Record deleted successfully 


### DROP TABLE:

In [19]:
# setzt geöffnet DB-Verbindung voraus:

try:
    mySql_Query = """DROP TABLE Laptop"""

    cursor = connection.cursor()
    result = cursor.execute(mySql_Query)
    print("Laptop Table deleted successfully ") 

except mysql.connector.Error as error:
    print("Failed to drop table {}".format(error))


Laptop Table deleted successfully 


### drop database electronic

In [20]:
# setzt geöffnet DB-Verbindung voraus:

try:
    sql = 'DROP DATABASE electronic'
    
    c = connection.cursor()
    c.execute(sql)
    print("Database electronic deleted successfully ") 

except mysql.connector.Error as error:
    print("Failed to drop database {}".format(error))
    

Database electronic deleted successfully 


### Verbindung schließen

In [21]:
# setzt geöffnet DB-Verbindung voraus:

if connection.is_connected():
    c.close()
    connection.close()
    print("Database connection closed")
    

Database connection closed


In [22]:
!docker stop mysqltest

mysqltest


In [23]:
!docker rm mysqltest

mysqltest


In [24]:
!docker ps -a

CONTAINER ID   IMAGE                 COMMAND                  CREATED             STATUS                        PORTS     NAMES
5669b6bb9007   dbeaver/cloudbeaver   "./run-server.sh"        About an hour ago   Exited (137) 36 minutes ago             mysql_dbeaver-client-1
b76e867fd85d   mysql                 "docker-entrypoint.sâ€¦"   About an hour ago   Exited (0) 36 minutes ago               mysql_dbeaver-db-1
71d32bf073a2   nginx_new             "/docker-entrypoint.â€¦"   7 days ago          Exited (0) 2 hours ago                  mynginx3
